# Day 3 - Exploratory Data Analysis
Bluestock Fintech Capstone | June 2026

15+ charts covering NAV trends, AUM growth, SIP flows, investor demographics, geographic distribution, sector allocation, and return correlations.


In [ ]:
import pathlib
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import Image, display

PROC   = pathlib.Path.cwd() / "data" / "processed"
CHARTS = pathlib.Path.cwd() / "reports" / "charts"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

df_nav   = pd.read_csv(PROC / "clean_nav_history.csv",         parse_dates=["date"])
df_fund  = pd.read_csv(PROC / "clean_fund_master.csv")
df_aum   = pd.read_csv(PROC / "clean_aum_by_fund_house.csv",   parse_dates=["date"])
df_sip   = pd.read_csv(PROC / "clean_monthly_sip_inflows.csv", parse_dates=["month"])
df_cat   = pd.read_csv(PROC / "clean_category_inflows.csv",    parse_dates=["month"])
df_tx    = pd.read_csv(PROC / "clean_investor_transactions.csv", parse_dates=["transaction_date"])
df_folio = pd.read_csv(PROC / "clean_industry_folio_count.csv",  parse_dates=["month"])
df_port  = pd.read_csv(PROC / "clean_portfolio_holdings.csv")

nav = df_nav.merge(df_fund[["amfi_code","scheme_name","sub_category","fund_house"]], on="amfi_code")
print("data loaded")


## Chart 1 — NAV Trend (Large Cap, Indexed to 100)

In [ ]:
lc_direct = df_fund[(df_fund["sub_category"]=="Large Cap") & (df_fund["plan"]=="Direct")]["amfi_code"].unique()
subset = nav[nav["amfi_code"].isin(lc_direct)].copy()
base   = subset.groupby("amfi_code")["nav"].transform("first")
subset["nav_idx"] = subset["nav"] / base * 100

fig = go.Figure()
for code, grp in subset.groupby("amfi_code"):
    name = grp["scheme_name"].iloc[0].split(" - ")[0]
    fig.add_trace(go.Scatter(x=grp["date"], y=grp["nav_idx"], mode="lines",
                             name=name, line=dict(width=1.5)))

fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.07,
              line_width=0, annotation_text="2023 Bull Run", annotation_position="top left")
fig.add_vrect(x0="2024-06-01", x1="2024-10-31", fillcolor="red", opacity=0.07,
              line_width=0, annotation_text="2024 Correction", annotation_position="top left")

fig.update_layout(title="Large Cap Fund NAV — Indexed to 100 (Jan 2022)",
                  xaxis_title="Date", yaxis_title="NAV Index",
                  height=480, template="plotly_white",
                  legend=dict(font=dict(size=9)))
fig.show()


## Chart 2 — AUM Growth by Fund House (2022–2025)

In [ ]:
df_aum["year"] = df_aum["date"].dt.year
yearly = df_aum.groupby(["year","fund_house"])["aum_lakh_crore"].max().reset_index()
yearly = yearly[yearly["year"].between(2022,2025)]
order  = (yearly[yearly["year"]==2025].sort_values("aum_lakh_crore", ascending=False)["fund_house"].tolist())

fig, ax = plt.subplots(figsize=(13,6))
yearly.pivot(index="fund_house", columns="year", values="aum_lakh_crore").reindex(order).plot(kind="bar", ax=ax, width=0.7)
ax.set_title("AUM by Fund House — 2022 to 2025 (Rs. lakh crore)", fontsize=13)
ax.set_xlabel(""); ax.set_ylabel("AUM (Rs. lakh crore)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="Year")
plt.tight_layout(); plt.show()


## Chart 3 — Monthly SIP Inflows (with ATH annotation)

In [ ]:
ath = df_sip.loc[df_sip["sip_inflow_crore"].idxmax()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_sip["month"], y=df_sip["sip_inflow_crore"],
                         mode="lines+markers", line=dict(color="steelblue", width=2),
                         marker=dict(size=4)))
fig.add_annotation(x=ath["month"], y=ath["sip_inflow_crore"],
                   text=f"ATH: Rs.{int(ath['sip_inflow_crore']):,} Cr  Dec 2025",
                   showarrow=True, arrowhead=2, ax=50, ay=-40,
                   font=dict(color="darkred", size=11))
fig.update_layout(title="Monthly SIP Inflows — Jan 2022 to Dec 2025",
                  xaxis_title="Month", yaxis_title="SIP Inflow (Rs. crore)",
                  template="plotly_white", height=420)
fig.show()


## Chart 4 — Category Inflow Heatmap

In [ ]:
df_cat["month_str"] = df_cat["month"].dt.strftime("%b %Y")
months_ordered = df_cat.sort_values("month")["month"].dt.strftime("%b %Y").unique()
pivot = df_cat.pivot(index="category", columns="month_str", values="net_inflow_crore")[months_ordered]

fig, ax = plt.subplots(figsize=(16,6))
sns.heatmap(pivot, ax=ax, cmap="RdYlGn", center=0, linewidths=0.3,
            cbar_kws={"label": "Net Inflow (Rs. crore)"})
ax.set_title("Category-wise Net Inflows — Monthly Heatmap", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout(); plt.show()


## Chart 5 — Investor Demographics (Age, SIP Amount, Gender)

In [ ]:
sip_only = df_tx[df_tx["transaction_type"]=="SIP"]
age_order = ["18-25","26-35","36-45","46-55","56+"]

fig, axes = plt.subplots(1, 3, figsize=(15,5))

age_counts = df_tx["age_group"].value_counts().reindex(age_order)
axes[0].pie(age_counts, labels=age_counts.index, autopct="%1.1f%%",
            startangle=140, colors=sns.color_palette("pastel"))
axes[0].set_title("Age Distribution")

sip_only.boxplot(column="amount_inr", by="age_group",
                 ax=axes[1], positions=range(len(age_order)), showfliers=False)
axes[1].set_xticklabels(age_order)
axes[1].set_title("SIP Amount by Age Group"); axes[1].set_xlabel(""); plt.suptitle("")

gender = df_tx["gender"].value_counts()
axes[2].pie(gender, labels=gender.index, autopct="%1.1f%%",
            colors=["#5B9BD5","#ED7D31"], startangle=90)
axes[2].set_title("Gender Split")

plt.suptitle("Investor Demographics", fontsize=14)
plt.tight_layout(); plt.show()


## Chart 6 — Geographic Distribution (State SIP + T30 vs B30)

In [ ]:
sip_only = df_tx[df_tx["transaction_type"]=="SIP"]
state_sip = sip_only.groupby("state")["amount_inr"].sum().sort_values() / 1e6

fig, axes = plt.subplots(1, 2, figsize=(14,5))
state_sip.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("SIP Amount by State"); axes[0].set_xlabel("Rs. millions")

tier = df_tx["city_tier"].value_counts()
axes[1].pie(tier, labels=tier.index, autopct="%1.1f%%",
            colors=["#2196F3","#FF9800"], startangle=90)
axes[1].set_title("T30 vs B30 City Tier")

plt.tight_layout(); plt.show()


## Chart 7 — Industry Folio Count Growth

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(df_folio["month"], df_folio["total_folios_crore"],
        color="steelblue", linewidth=2, marker="o", markersize=4)
ax.fill_between(df_folio["month"], df_folio["total_folios_crore"], alpha=0.15, color="steelblue")

for milestone in [20.0, 25.0]:
    row = df_folio[df_folio["total_folios_crore"] >= milestone].iloc[0]
    ax.annotate(f"{milestone} Cr", xy=(row["month"], milestone),
                xytext=(10,10), textcoords="offset points", fontsize=9, color="darkgreen",
                arrowprops=dict(arrowstyle="->", color="darkgreen"))

ax.set_title("Industry Folio Count Growth — Jan 2022 to Dec 2025", fontsize=13)
ax.set_ylabel("Total Folios (crore)"); ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f Cr"))
plt.tight_layout(); plt.show()


## Chart 8 — NAV Return Correlation Matrix (Large Cap Funds)

In [ ]:
lc_direct = df_fund[(df_fund["sub_category"]=="Large Cap") & (df_fund["plan"]=="Direct")]["amfi_code"].unique()[:10]
pivot   = df_nav[df_nav["amfi_code"].isin(lc_direct)].pivot(index="date", columns="amfi_code", values="nav")
returns = pivot.pct_change().dropna()

code_name = df_fund.set_index("amfi_code")["scheme_name"].to_dict()
short     = {c: code_name[c].split(" - ")[0].replace(" Fund","").strip() for c in lc_direct if c in code_name}
returns.rename(columns=short, inplace=True)
corr = returns.corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=0.5, vmax=1.0, linewidths=0.5)
ax.set_title("Daily Return Correlation — Large Cap Funds", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
plt.tight_layout(); plt.show()


## Chart 9 — Sector Allocation Donut

In [ ]:
sector_wt = df_port.groupby("sector")["weight_pct"].sum().sort_values(ascending=False)
threshold = sector_wt.sum() * 0.03
main  = sector_wt[sector_wt >= threshold].copy()
other = sector_wt[sector_wt < threshold].sum()
if other > 0:
    main["Other"] = other

fig, ax = plt.subplots(figsize=(9,7))
ax.pie(main, labels=main.index, autopct="%1.1f%%", pctdistance=0.82, startangle=140,
       colors=sns.color_palette("tab10", len(main)), wedgeprops=dict(width=0.5))
ax.set_title("Sector Allocation — Aggregate Across All Equity Funds", fontsize=12)
plt.tight_layout(); plt.show()


## Chart 10 — SIP Inflow vs Active Accounts (Dual Axis)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,5))
ax2 = ax1.twinx()

ax1.bar(df_sip["month"], df_sip["sip_inflow_crore"],
        color="steelblue", alpha=0.6, width=20, label="SIP Inflow (crore)")
ax2.plot(df_sip["month"], df_sip["active_sip_accounts_crore"],
         color="darkorange", linewidth=2, marker="o", markersize=3, label="Active Accounts (crore)")

ax1.set_title("SIP Inflow vs Active Accounts — 2022 to 2025", fontsize=13)
ax1.set_ylabel("SIP Inflow (Rs. crore)", color="steelblue")
ax2.set_ylabel("Active Accounts (crore)", color="darkorange")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2.tick_params(axis="y", labelcolor="darkorange")

lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()


## 10 Key EDA Findings

1. **NAV growth sustained despite corrections** — Large cap funds delivered 80-120% indexed returns from Jan 2022 to May 2026. The 2024 mid-year correction reversed quickly, with most funds recovering within 3 months. (Chart 1)

2. **SBI Mutual Fund dominates AUM** — SBI leads all fund houses at Rs.12.5 lakh crore AUM in 2025, nearly 2x ICICI Prudential (Rs.10.74L Cr) and consistently widened this gap every year since 2022. (Chart 2)

3. **SIP inflows on a structural uptrend** — Monthly SIP inflows grew from ~Rs.11,500 Cr in Jan 2022 to an all-time high of Rs.31,002 Cr in Dec 2025, a 2.7x increase in under 4 years. (Chart 3)

4. **Equity categories drive most inflows** — Large Cap and Flexi Cap categories show consistently positive inflows across all months on the heatmap. Liquid funds show high volatility with large negative net outflows in some months, which is normal due to institutional redemptions. (Chart 4)

5. **26-35 age group is the dominant SIP investor** — This cohort represents the largest share of transactions. Median SIP amount is relatively uniform across age groups (~Rs.10,900-11,600), suggesting SIP is used broadly regardless of age. (Chart 5)

6. **Punjab and Tamil Nadu lead SIP contributions** — Both states outperform larger states like Maharashtra and Delhi in total SIP amount, likely due to higher participation rates in smaller cities. T30 cities still account for the majority of transaction volume but B30 is catching up. (Chart 6)

7. **Folio count doubled in 4 years** — Industry folios grew from 13.26 Cr to 26.12 Cr between Jan 2022 and Dec 2025, crossing the 20 Cr and 25 Cr milestones in quick succession in 2024. (Chart 7)

8. **Large cap fund returns are highly correlated** — Pairwise return correlations among large cap funds range from 0.85 to 0.98, meaning most large cap funds move almost identically. This limits diversification benefit from holding multiple large cap funds. (Chart 8)

9. **Banking and IT dominate equity portfolios** — Banking sector has the highest aggregate weight across all equity funds, followed by IT and Consumer Goods. This concentration means fund performance is heavily linked to Nifty Bank and Nifty IT indices. (Chart 9)

10. **Active SIP accounts grew faster than inflow growth** — Active accounts rose from ~3.5 Cr to 9.35 Cr while inflows grew 2.7x. This means average SIP amount per account increased too, suggesting both more investors joining AND existing investors increasing SIP amounts. (Chart 10)
